# 04c — share_cana_eq52 pré-2018 (B4.M.3)

**Pré-registro v2.3.9 §9.6**

Constrói o `share_cana_eq52` pré-tratamento (janela estrita 2012-2017) a partir da PAM Tabela 1612, para alimentar a heterogeneidade §9.6 (configuração D) na fase B4.M.5.

Fórmula (Eq. 52 SEEG, restrita à janela pré-RenovaBio):

$$\overline{share\_cana\_eq52}_i^{pre} = \frac{1}{|T_{pre}|}\sum_{t=2012}^{2017}\frac{Cana_{it}}{Cana_{it}+Milho_{it}+Algodao_{it}}$$

**Sem soja** no denominador (leguminosa fixadora de N, não pesa na Eq. 52 — ver Apêndice H Parte II §6).

**Pré-condições no Drive:**
- `pipeline/build_share_cana_pre2018.py` (módulo)
- `data/interim/pam_1612_long_2012_2024.parquet` (parsing D2)

## Setup

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/Renovabio - EcoEco")
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
print("setup ok")

Mounted at /content/drive
setup ok


In [5]:
import importlib
from pipeline import config
importlib.reload(config)
from pipeline.config import interim, out_pre, PARAMS

# build_share_cana_pre2018.py deve estar subido em pipeline/
from pipeline import build_share_cana_pre2018 as b3
importlib.reload(b3)
print("builder B4.M.3 carregado")
print("janela pre:", b3.PRE_YEARS)

builder B4.M.3 carregado
janela pre: [2012, 2013, 2014, 2015, 2016, 2017]


## Executa B4.M.3

In [6]:
pam = pd.read_parquet(interim("pam_1612_long_2012_2024.parquet"))
print("PAM:", pam.shape)

df_share, df_quality = b3.build_share_cana_pre2018(pam)
print("share pre-2018:", df_share.shape)
df_quality.T

PAM: (289224, 10)
share pre-2018: (2018, 10)


,0
janela_pre,2012-2017
n_municipios_canavieiros_cs,2018
n_municipios_pam_total,5562
share_mediana,0.828309
share_p25,0.290493
share_p50,0.828309
share_p75,0.984439
share_p90,0.997842
n_share_gt_050,1356
n_share_gt_075,1124


In [7]:
q = df_quality.iloc[0]
print("Canavieiros CS (2012-2017):", q["n_municipios_canavieiros_cs"])
print(f"mediana={q['share_mediana']:.4f}  P25={q['share_p25']:.3f}  "
      f"P50={q['share_p50']:.3f}  P75={q['share_p75']:.3f}")
print("\nSubsets ex-ante 9.6.2:")
print(f"  cana_dominante  (>P50): {q['n_cana_dominante_gtP50']}")
print(f"  cana_minoritaria(<P25): {q['n_cana_minoritaria_ltP25']}")
print(df_share["subset_9_6"].value_counts().to_string())

Canavieiros CS (2012-2017): 2018
mediana=0.8283  P25=0.290  P50=0.828  P75=0.984

Subsets ex-ante 9.6.2:
  cana_dominante  (>P50): 1009
  cana_minoritaria(<P25): 505
subset_9_6
cana_dominante      1009
cana_minoritaria     505
intermediario        504


In [8]:
df_share.to_csv(interim("share_cana_eq52_pre2018.csv"), index=False)
df_quality.to_csv(out_pre("share_cana_eq52_pre2018_quality.csv"), index=False)
print("OK share_cana_eq52_pre2018.csv salvo em data/interim/")
print("OK quality report salvo em outputs_pre/")
df_share.head(10)

OK share_cana_eq52_pre2018.csv salvo em data/interim/
OK quality report salvo em outputs_pre/


,cod_ibge,municipio_uf,uf,share_cana_eq52_pre,n_anos_validos,cana_t_mean,milho_t_mean,algodao_t_mean,denom_t_mean,subset_9_6
0,5200050,Abadia de Goiás (GO),GO,0.166667,6,5.000000e+01,1133.333333,0.000000,1.183333e+03,cana_minoritaria
1,5200100,Abadiânia (GO),GO,0.024691,6,2.333333e+02,10021.333333,0.000000,1.025467e+04,cana_minoritaria
2,5200134,Acreúna (GO),GO,0.968290,6,1.828387e+06,57438.333333,2395.166667,1.888220e+06,cana_dominante
3,5200308,Alexânia (GO),GO,0.392102,6,7.632500e+03,9716.666667,0.000000,1.734917e+04,intermediario
4,5200555,Alto Horizonte (GO),GO,0.428747,6,2.666667e+02,425.000000,0.000000,6.916667e+02,intermediario
5,5200605,Alto Paraíso de Goiás (GO),GO,0.033545,6,5.500000e+02,15908.000000,0.000000,1.645800e+04,cana_minoritaria
6,5200803,Alvorada do Norte (GO),GO,0.138709,6,1.265833e+03,6637.500000,0.000000,7.903333e+03,cana_minoritaria
7,5200829,Amaralina (GO),GO,0.210565,6,1.913333e+02,1013.333333,0.000000,1.204667e+03,cana_minoritaria
8,5200852,Americano do Brasil (GO),GO,0.636223,6,1.613797e+05,4014.166667,0.000000,1.653938e+05,intermediario
9,5200902,Amorinópolis (GO),GO,0.759792,6,6.400000e+02,314.166667,0.000000,9.541667e+02,intermediario


In [9]:
# Sanity: correlacao com share all-years (se existir do D2)
try:
    ay = pd.read_csv(interim("share_cana_eq52_municipio.csv"))
    m = df_share.assign(cod_ibge=df_share["cod_ibge"].astype(str)).merge(
        ay.assign(cod_ibge=ay["cod_ibge"].astype(str))[
            ["cod_ibge","share_cana_eq52_mean"]], on="cod_ibge")
    print(f"municipios em comum: {len(m)}")
    print(f"correlacao pre vs all-years: "
          f"{m['share_cana_eq52_pre'].corr(m['share_cana_eq52_mean']):.4f}")
    print("(alta esperada; difere porque janela pre exclui 2018-2024)")
except Exception as e:
    print("sanity pulado:", e)

sanity pulado: [Errno 2] No such file or directory: '/content/drive/MyDrive/Renovabio - EcoEco/data/interim/share_cana_eq52_municipio.csv'


## Conclusão B4.M.3

Se rodou sem erro:
- `share_cana_eq52_pre2018.csv` — município × share + subset §9.6.2 pré-marcado
- quality report com cortes P25/P50 ex-ante

**Próximo:** B4.M.4 (CS-DR × 5 sub-canais — notebook 11d). Já entregue.